# PDA 26/27 · Практика 1 · Марюшкина · Чуева

## Подготовка

In [ ]:
# Установка библиотек (~1 минута)
!pip install -q "pyspark==4.2.*" "polars>=1.40" "dask[dataframe]" gdown

In [2]:
import os, re, subprocess

def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

if java_major() < 17:
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Java", java_major())

Java 17


In [3]:
# Данные: BTS On-Time Performance, 2023–2024
import glob, gdown

FILE_IDS = ['1NQuold7jvGZfHikbvtj0QynbhM90pT8L', '1HNAGhPHE6BmamcI0D767-BFhF4BWs6Hh', '1ERgltJRuybLl7sIUUVckzpUVzyP1ttCq', '1XrUk6ZYwQLQTjtsMHKxzzERP4dCLMEao', '1qy95Qpd93VClYPethMtMDJdAMBl21fnW', '1js5pbc_xe6kP7W4_lp6HaeXR0oAw2ZDL', '1qb4_uKGy0KU_4jRzaauqeObaaCQMtdBZ', '1TuC71TgrJDgpOvQVKl9jn89fAr9El1Pa', '1nt7e9Pszf2-IB3GsKHNRHZ_SPzxS0E2L', '1V5tCiD9jRCss0KoJxk-r1zOCw3onwez3', '1WsKGiVBV2p-sgiDePT6HZKUezWPc2zUH']
N_FILES = len(FILE_IDS)

if len(glob.glob("T_ONTIME_REPORTING*.csv")) < N_FILES:
    for fid in FILE_IDS[:N_FILES]:
        gdown.download(id=fid, quiet=True)

FILES = sorted(glob.glob("T_ONTIME_REPORTING*.csv"))[:N_FILES]
CSV_GLOB = "T_ONTIME_REPORTING*.csv"
print(len(FILES), "файлов:", FILES[:3], "...")

11 файлов: ['T_ONTIME_REPORTING 10.csv', 'T_ONTIME_REPORTING 11.csv', 'T_ONTIME_REPORTING 2.csv'] ...


In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PDA-practice1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/16 20:12:29 WARN Utils: Your hostname, MacBook-Air-Maruskina.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.243 instead (on interface en0)
26/10/16 20:12:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/Users/sofa_maruysh/Desktop/PDA/.venv-practice1/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/16 20:12:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
schema = T.StructType([
    T.StructField("YEAR", T.IntegerType()),
    T.StructField("FL_DATE", T.StringType()),
    T.StructField("OP_UNIQUE_CARRIER", T.StringType()),
    T.StructField("ORIGIN_AIRPORT_ID", T.IntegerType()),
    T.StructField("ORIGIN", T.StringType()),
    T.StructField("ORIGIN_CITY_NAME", T.StringType()),
    T.StructField("DEST_AIRPORT_ID", T.IntegerType()),
    T.StructField("DEST", T.StringType()),
    T.StructField("DEST_CITY_NAME", T.StringType()),
    T.StructField("DEP_DELAY", T.DoubleType()),
    T.StructField("ARR_DELAY", T.DoubleType()),
    T.StructField("CANCELLED", T.DoubleType()),
    T.StructField("ACTUAL_ELAPSED_TIME", T.DoubleType()),
    T.StructField("DISTANCE", T.DoubleType()),
    T.StructField("CARRIER_DELAY", T.DoubleType()),
    T.StructField("WEATHER_DELAY", T.DoubleType()),
    T.StructField("NAS_DELAY", T.DoubleType()),
    T.StructField("SECURITY_DELAY", T.DoubleType()),
    T.StructField("LATE_AIRCRAFT_DELAY", T.DoubleType()),
])

import shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import dask.dataframe as dd

def timeit(action, label):
    start = time.perf_counter()
    result = action()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.2f} с")
    return result, elapsed

def check_series_equal(series_by_name, rtol=1e-7, atol=1e-7):
    prepared = {name: value.sort_index().astype(float) for name, value in series_by_name.items()}
    first_name = next(iter(prepared))
    first = prepared[first_name]
    for name, current in list(prepared.items())[1:]:
        if not first.index.equals(current.index):
            raise AssertionError(f"Наборы групп не совпали: {first_name} и {name}")
        np.testing.assert_allclose(first.values, current.values, rtol=rtol, atol=atol)
    print("Проверка пройдена: все результаты совпадают.")

## Задача 1

Объедините все CSV-файлы в одну таблицу, сгруппируйте по перевозчику (`OP_UNIQUE_CARRIER`) и посчитайте суммарное расстояние (`DISTANCE`). Сделайте это в **pandas**, **Dask**, **Polars** и **PySpark** и измерьте время каждого варианта вместе с чтением данных.

1. Убедитесь, что все четыре ответа совпадают.
2. Какой инструмент оказался быстрее всех и почему? Почему PySpark на этих данных не лидер?
3. Как изменится время pandas, если читать только две нужные колонки (`usecols`)?

В Spark и Polars можно читать сразу все файлы по шаблону `CSV_GLOB`; в pandas — использовать `pd.concat`.

In [6]:
#общий Spark DataFrame используется и в следующих задачах
flights = (
    spark.read.option("header", True).schema(schema).csv(CSV_GLOB)
    .withColumn("FL_DATE", F.to_date("FL_DATE", "M/d/yyyy h:mm:ss a"))
    .withColumn("MONTH", F.month("FL_DATE"))
)

def pandas_all_columns():
    frame = pd.concat((pd.read_csv(file, low_memory=False) for file in FILES), ignore_index=True)
    return frame.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum()

def dask_all_columns():
    frame = dd.read_csv(FILES, assume_missing=True)
    return frame.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum().compute()

def polars_all_columns():
    return (
        pl.scan_csv(CSV_GLOB)
        .group_by("OP_UNIQUE_CARRIER")
        .agg(pl.col("DISTANCE").sum())
        .collect()
    )

def spark_all_columns():
    return flights.groupBy("OP_UNIQUE_CARRIER").agg(F.sum("DISTANCE").alias("DISTANCE")).collect()

pandas_result, pandas_time = timeit(pandas_all_columns, "pandas, все колонки")
dask_result, dask_time = timeit(dask_all_columns, "Dask")
polars_result, polars_time = timeit(polars_all_columns, "Polars")
spark_result, spark_time = timeit(spark_all_columns, "PySpark")

polars_series = polars_result.to_pandas().set_index("OP_UNIQUE_CARRIER")["DISTANCE"]
spark_series = pd.Series(
    {row["OP_UNIQUE_CARRIER"]: row["DISTANCE"] for row in spark_result},
    name="DISTANCE",
)
check_series_equal({
    "pandas": pandas_result,
    "Dask": dask_result,
    "Polars": polars_series,
    "PySpark": spark_series,
})

def pandas_two_columns():
    frame = pd.concat(
        (pd.read_csv(file, usecols=["OP_UNIQUE_CARRIER", "DISTANCE"]) for file in FILES),
        ignore_index=True,
    )
    return frame.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum()

pandas_two_result, pandas_two_time = timeit(pandas_two_columns, "pandas, только 2 колонки")
check_series_equal({"pandas (все)": pandas_result, "pandas (2 колонки)": pandas_two_result})

times = {"pandas": pandas_time, "Dask": dask_time, "Polars": polars_time, "PySpark": spark_time}
fastest = min(times, key=times.get)
print(f"\nБыстрее всех: {fastest} ({times[fastest]:.2f} с).")
print("Polars обычно выигрывает на одной машине благодаря многопоточному движку и оптимизации запроса.")
print("PySpark здесь не лидер из-за затрат на запуск JVM, планирование задач и сериализацию; данных мало для окупаемости этих расходов.")
print(f"Чтение двух колонок ускорило pandas в {pandas_time / pandas_two_time:.2f} раза.")

pandas, все колонки: 5.16 с


Dask: 2.91 с
Polars: 0.18 с


PySpark: 2.30 с
Проверка пройдена: все результаты совпадают.


pandas, только 2 колонки: 2.04 с
Проверка пройдена: все результаты совпадают.

Быстрее всех: Polars (0.18 с).
Polars обычно выигрывает на одной машине благодаря многопоточному движку и оптимизации запроса.
PySpark здесь не лидер из-за затрат на запуск JVM, планирование задач и сериализацию; данных мало для окупаемости этих расходов.
Чтение двух колонок ускорило pandas в 2.53 раза.


## Задача 2. Кэширование

Отберите задержанные рейсы (`ARR_DELAY > 15`) и посчитайте по ним три разных агрегата: число рейсов по месяцам, среднюю задержку по перевозчикам и топ-5 направлений. Измерьте общее время без кэширования и с `cache()` отобранного DataFrame.

In [7]:
def three_aggregates(frame):
    by_month = frame.groupBy("MONTH").count().orderBy("MONTH").collect()
    avg_by_carrier = (
        frame.groupBy("OP_UNIQUE_CARRIER")
        .agg(F.avg("ARR_DELAY").alias("AVG_ARR_DELAY"))
        .orderBy("OP_UNIQUE_CARRIER")
        .collect()
    )
    top_routes = (
        frame.withColumn("ROUTE", F.concat_ws(" → ", "ORIGIN", "DEST"))
        .groupBy("ROUTE").count()
        .orderBy(F.desc("count"), "ROUTE")
        .limit(5)
        .collect()
    )
    return by_month, avg_by_carrier, top_routes

def without_cache():
    delayed = flights.filter(F.col("ARR_DELAY") > 15)
    return three_aggregates(delayed)

def with_cache():
    delayed = flights.filter(F.col("ARR_DELAY") > 15).cache()
    delayed.count()  #материализую кэш; это время также входит в замер
    result = three_aggregates(delayed)
    delayed.unpersist()
    return result

uncached_result, uncached_time = timeit(without_cache, "Три агрегата без кэша")
cached_result, cached_time = timeit(with_cache, "Три агрегата с кэшем")
assert uncached_result == cached_result

print("Проверка пройдена: результаты с кэшем и без него совпадают.")
print("Рейсы по месяцам:", uncached_result[0])
print("Средняя задержка по перевозчикам:", uncached_result[1])
print("Топ-5 направлений:", uncached_result[2])
print(f"Ускорение от кэширования: {uncached_time / cached_time:.2f} раза.")
print("Кэш полезен, потому что один и тот же отфильтрованный набор используется в трех действиях.")

Три агрегата без кэша: 4.21 с


Три агрегата с кэшем: 3.55 с
Проверка пройдена: результаты с кэшем и без него совпадают.
Рейсы по месяцам: [Row(MONTH=1, count=235398), Row(MONTH=2, count=89724), Row(MONTH=3, count=131292), Row(MONTH=4, count=121744), Row(MONTH=5, count=100446), Row(MONTH=6, count=147875), Row(MONTH=7, count=162632), Row(MONTH=8, count=124261), Row(MONTH=11, count=73004), Row(MONTH=12, count=85517)]
Средняя задержка по перевозчикам: [Row(OP_UNIQUE_CARRIER='9E', AVG_ARR_DELAY=79.62709112193649), Row(OP_UNIQUE_CARRIER='AA', AVG_ARR_DELAY=88.46872092386843), Row(OP_UNIQUE_CARRIER='AS', AVG_ARR_DELAY=53.13484190228376), Row(OP_UNIQUE_CARRIER='B6', AVG_ARR_DELAY=85.15718237635787), Row(OP_UNIQUE_CARRIER='DL', AVG_ARR_DELAY=73.99453830909275), Row(OP_UNIQUE_CARRIER='F9', AVG_ARR_DELAY=80.77615612014966), Row(OP_UNIQUE_CARRIER='G4', AVG_ARR_DELAY=84.7257975493984), Row(OP_UNIQUE_CARRIER='HA', AVG_ARR_DELAY=49.29736455152132), Row(OP_UNIQUE_CARRIER='MQ', AVG_ARR_DELAY=62.37266382622788), Row(OP_UNIQUE_CARRIER

## Задача 3. Spark SQL

1. **SQL.** Для каждого месяца найдите 3 аэропорта вылета с наибольшей долей отмененных рейсов среди аэропортов с не менее чем 1000 вылетов в этом месяце.
2. **DataFrame API.** Решите ту же задачу без SQL и убедитесь, что ответы совпадают.
3. Для каждого перевозчика найдите день с максимальной средней задержкой прилета.
4. Посчитайте ежедневное число рейсов и его изменение относительно предыдущего дня (`lag`). В какой день был самый большой спад?

In [8]:
flights.createOrReplaceTempView("flights")

# 1. Решение с помощью Spark SQL
top_airports_sql = spark.sql("""
WITH airport_stats AS (
    SELECT
        MONTH,
        ORIGIN,
        COUNT(*) AS flights_count,
        AVG(CANCELLED) AS cancelled_share
    FROM flights
    GROUP BY MONTH, ORIGIN
    HAVING COUNT(*) >= 1000
), ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY MONTH
               ORDER BY cancelled_share DESC, ORIGIN
           ) AS place
    FROM airport_stats
)
SELECT MONTH, ORIGIN, flights_count, cancelled_share, place
FROM ranked
WHERE place <= 3
ORDER BY MONTH, place
""")

# 2. То же самое с помощью DataFrame API
airport_stats = (
    flights.groupBy("MONTH", "ORIGIN")
    .agg(
        F.count("*").alias("flights_count"),
        F.avg("CANCELLED").alias("cancelled_share"),
    )
    .filter(F.col("flights_count") >= 1000)
)
airport_window = Window.partitionBy("MONTH").orderBy(F.desc("cancelled_share"), "ORIGIN")
top_airports_df = (
    airport_stats.withColumn("place", F.row_number().over(airport_window))
    .filter(F.col("place") <= 3)
    .orderBy("MONTH", "place")
)

sql_rows = top_airports_sql.collect()
df_rows = top_airports_df.collect()
assert sql_rows == df_rows
print("SQL и DataFrame API дали одинаковый ответ.")
top_airports_sql.show(36, truncate=False)

# 3. День с максимальной средней задержкой для каждого перевозчика
carrier_day_stats = (
    flights.groupBy("OP_UNIQUE_CARRIER", "FL_DATE")
    .agg(F.avg("ARR_DELAY").alias("AVG_ARR_DELAY"))
    .filter(F.col("AVG_ARR_DELAY").isNotNull())
)
carrier_window = Window.partitionBy("OP_UNIQUE_CARRIER").orderBy(
    F.desc("AVG_ARR_DELAY"), "FL_DATE"
)
worst_day_by_carrier = (
    carrier_day_stats.withColumn("place", F.row_number().over(carrier_window))
    .filter(F.col("place") == 1)
    .drop("place")
    .orderBy("OP_UNIQUE_CARRIER")
)
print("День с максимальной средней задержкой:")
worst_day_by_carrier.show(truncate=False)

# 4. Число рейсов по дням и изменение относительно предыдущего дня
daily_counts = flights.groupBy("FL_DATE").agg(F.count("*").alias("flights_count"))
date_window = Window.orderBy("FL_DATE")
daily_change = (
    daily_counts
    .withColumn("previous_day_count", F.lag("flights_count").over(date_window))
    .withColumn("change", F.col("flights_count") - F.col("previous_day_count"))
    .orderBy("FL_DATE")
)
biggest_drop = (
    daily_change
    .filter(F.col("change").isNotNull())
    .orderBy(F.asc("change"))
    .first()
)
print("Самый большой спад относительно предыдущего дня:", biggest_drop)
daily_change.show(10, truncate=False)

SQL и DataFrame API дали одинаковый ответ.


+-----+------+-------------+--------------------+-----+
|MONTH|ORIGIN|flights_count|cancelled_share     |place|
+-----+------+-------------+--------------------+-----+
|1    |ASE   |1858         |0.1377825618945102  |1    |
|1    |EUG   |1096         |0.08394160583941605 |2    |
|1    |BUF   |3165         |0.07551342812006319 |3    |
|2    |PDX   |4263         |0.050668543279380716|1    |
|2    |DAL   |5444         |0.048493754592211606|2    |
|2    |MKE   |1857         |0.04577275175013463 |3    |
|3    |ALB   |1041         |0.03842459173871278 |1    |
|3    |SYR   |1125         |0.035555555555555556|2    |
|3    |ANC   |1292         |0.02631578947368421 |3    |
|4    |FLL   |8071         |0.08462396233428324 |1    |
|4    |EWR   |12022        |0.037015471635335216|2    |
|4    |JFK   |11638        |0.030761299192301084|3    |
|5    |ORF   |1800         |0.025555555555555557|1    |
|5    |HNL   |5166         |0.02032520325203252 |2    |
|5    |KOA   |1457         |0.017844886753603295

+-----------------+----------+------------------+
|OP_UNIQUE_CARRIER|FL_DATE   |AVG_ARR_DELAY     |
+-----------------+----------+------------------+
|9E               |2024-01-16|82.2046511627907  |
|AA               |2023-01-11|84.73424036281179 |
|AS               |2024-01-17|44.43098591549296 |
|B6               |2023-07-16|131.88758782201404|
|DL               |2024-01-09|47.38364211477632 |
|F9               |2023-01-02|57.10070257611241 |
|G4               |2024-01-16|141.125           |
|HA               |2023-05-12|164.48314606741573|
|MQ               |2024-01-15|81.93050193050193 |
|NK               |2023-06-01|135.52140672782875|
|OH               |2024-01-16|99.97507788161994 |
|OO               |2024-01-15|57.193423597678915|
|UA               |2023-06-27|87.80461329715061 |
|WN               |2023-01-11|88.93786707882535 |
|YX               |2024-01-16|87.76817288801571 |
+-----------------+----------+------------------+



Самый большой спад относительно предыдущего дня: Row(FL_DATE=datetime.date(2023, 11, 23), flights_count=11417, previous_day_count=20080, change=-8663)


+----------+-------------+------------------+------+
|FL_DATE   |flights_count|previous_day_count|change|
+----------+-------------+------------------+------+
|2023-01-01|15856        |NULL              |NULL  |
|2023-01-02|18075        |15856             |2219  |
|2023-01-03|17683        |18075             |-392  |
|2023-01-04|16989        |17683             |-694  |
|2023-01-05|17021        |16989             |32    |
|2023-01-06|17087        |17021             |66    |
|2023-01-07|15589        |17087             |-1498 |
|2023-01-08|17773        |15589             |2184  |
|2023-01-09|18233        |17773             |460   |
|2023-01-10|17068        |18233             |-1165 |
+----------+-------------+------------------+------+
only showing top 10 rows


## Задача 4. Parquet

1. Сохраните `flights` в Parquet с партиционированием по `MONTH`. Сравните размер на диске с исходными CSV.
2. Прочитайте получившийся каталог в **Polars** (`pl.scan_parquet(..., hive_partitioning=True)`) и в **Dask** (`dd.read_parquet`).
3. Посчитайте среднюю задержку прилета по перевозчикам за декабрь по CSV и по Parquet во всех трех инструментах. Во сколько раз быстрее работает Parquet? Найдите в `explain()` Polars и Spark признаки того, что читаются только нужные колонки и папки.

In [9]:
PARQUET_DIR = "flights_parquet"
if Path(PARQUET_DIR).exists():
    shutil.rmtree(PARQUET_DIR)

flights.write.mode("overwrite").partitionBy("MONTH").parquet(PARQUET_DIR)

def path_size(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())

csv_size = sum(path_size(file) for file in FILES)
parquet_size = path_size(PARQUET_DIR)
print(f"CSV: {csv_size / 1024**2:.1f} МБ")
print(f"Parquet: {parquet_size / 1024**2:.1f} МБ")
print(f"Parquet занимает в {csv_size / parquet_size:.2f} раза меньше места.")

parquet_pattern = f"{PARQUET_DIR}/**/*.parquet"
polars_parquet = pl.scan_parquet(parquet_pattern, hive_partitioning=True)
dask_parquet = dd.read_parquet(PARQUET_DIR)
print("Parquet прочитан в Polars и Dask.")

def spark_csv_december():
    return (
        flights.filter(F.col("MONTH") == 12)
        .groupBy("OP_UNIQUE_CARRIER")
        .agg(F.avg("ARR_DELAY").alias("ARR_DELAY"))
        .collect()
    )

def spark_parquet_december():
    return (
        spark.read.parquet(PARQUET_DIR)
        .filter(F.col("MONTH") == 12)
        .groupBy("OP_UNIQUE_CARRIER")
        .agg(F.avg("ARR_DELAY").alias("ARR_DELAY"))
        .collect()
    )

def polars_csv_december():
    return (
        pl.scan_csv(CSV_GLOB)
        .filter(
            pl.col("FL_DATE")
            .str.to_datetime("%m/%d/%Y %I:%M:%S %p", strict=False)
            .dt.month() == 12
        )
        .group_by("OP_UNIQUE_CARRIER")
        .agg(pl.col("ARR_DELAY").mean())
        .collect()
    )

def polars_parquet_december():
    return (
        polars_parquet
        .filter(pl.col("MONTH") == 12)
        .group_by("OP_UNIQUE_CARRIER")
        .agg(pl.col("ARR_DELAY").mean())
        .collect()
    )

def dask_csv_december():
    frame = dd.read_csv(FILES, usecols=["FL_DATE", "OP_UNIQUE_CARRIER", "ARR_DELAY"], assume_missing=True)
    frame = frame.assign(
        MONTH=dd.to_datetime(
            frame["FL_DATE"], format="%m/%d/%Y %I:%M:%S %p"
        ).dt.month
    )
    return frame[frame["MONTH"] == 12].groupby("OP_UNIQUE_CARRIER")["ARR_DELAY"].mean().compute()

def dask_parquet_december():
    frame = dd.read_parquet(PARQUET_DIR, columns=["MONTH", "OP_UNIQUE_CARRIER", "ARR_DELAY"])
    return frame[frame["MONTH"] == 12].groupby("OP_UNIQUE_CARRIER")["ARR_DELAY"].mean().compute()

spark_csv_result, spark_csv_time = timeit(spark_csv_december, "Spark CSV")
spark_pq_result, spark_pq_time = timeit(spark_parquet_december, "Spark Parquet")
polars_csv_result, polars_csv_time = timeit(polars_csv_december, "Polars CSV")
polars_pq_result, polars_pq_time = timeit(polars_parquet_december, "Polars Parquet")
dask_csv_result, dask_csv_time = timeit(dask_csv_december, "Dask CSV")
dask_pq_result, dask_pq_time = timeit(dask_parquet_december, "Dask Parquet")

def spark_rows_to_series(rows):
    return pd.Series({row["OP_UNIQUE_CARRIER"]: row["ARR_DELAY"] for row in rows})

def polars_to_series(frame):
    return frame.to_pandas().set_index("OP_UNIQUE_CARRIER")["ARR_DELAY"]

check_series_equal({
    "Spark CSV": spark_rows_to_series(spark_csv_result),
    "Spark Parquet": spark_rows_to_series(spark_pq_result),
    "Polars CSV": polars_to_series(polars_csv_result),
    "Polars Parquet": polars_to_series(polars_pq_result),
    "Dask CSV": dask_csv_result,
    "Dask Parquet": dask_pq_result,
})

print(f"Spark: Parquet быстрее CSV в {spark_csv_time / spark_pq_time:.2f} раза.")
print(f"Polars: Parquet быстрее CSV в {polars_csv_time / polars_pq_time:.2f} раза.")
print(f"Dask: Parquet быстрее CSV в {dask_csv_time / dask_pq_time:.2f} раза.")

print("\nПлан Spark (ищем PartitionFilters и ReadSchema):")
(
    spark.read.parquet(PARQUET_DIR)
    .filter(F.col("MONTH") == 12)
    .select("OP_UNIQUE_CARRIER", "ARR_DELAY")
    .explain("formatted")
)

print("\nПлан Polars (проекция нужных столбцов и фильтр по MONTH):")
print(
    polars_parquet
    .filter(pl.col("MONTH") == 12)
    .select("OP_UNIQUE_CARRIER", "ARR_DELAY")
    .explain()
)

CSV: 732.8 МБ
Parquet: 61.3 МБ
Parquet занимает в 11.94 раза меньше места.
Parquet прочитан в Polars и Dask.


Spark CSV: 2.66 с


Spark Parquet: 0.22 с


Polars CSV: 0.21 с
Polars Parquet: 0.01 с


Dask CSV: 1.96 с
Dask Parquet: 0.10 с
Проверка пройдена: все результаты совпадают.
Spark: Parquet быстрее CSV в 11.93 раза.
Polars: Parquet быстрее CSV в 18.06 раза.
Dask: Parquet быстрее CSV в 20.04 раза.

План Spark (ищем PartitionFilters и ReadSchema):
== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [OP_UNIQUE_CARRIER#2788, ARR_DELAY#2796, MONTH#2805]
Batched: true
Location: InMemoryFileIndex [file:/Users/sofa_maruysh/Desktop/PDA/flights_parquet]
PartitionFilters: [isnotnull(MONTH#2805), (MONTH#2805 = 12)]
ReadSchema: struct<OP_UNIQUE_CARRIER:string,ARR_DELAY:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [OP_UNIQUE_CARRIER#2788, ARR_DELAY#2796, MONTH#2805]

(3) Project [codegen id : 1]
Output [2]: [OP_UNIQUE_CARRIER#2788, ARR_DELAY#2796]
Input [3]: [OP_UNIQUE_CARRIER#2788, ARR_DELAY#2796, MONTH#2805]



План Polars (проекция нужных столбцов и фильтр по MONTH):
simple π 2/2 ["OP_UNIQUE_CARRIER", ... 1 other col

## Задача 5. Партиции в Dask

1. Прочитайте CSV в Dask с `blocksize` 16 МБ, 64 МБ и 256 МБ. Сколько получается партиций?
2. Для каждого варианта измерьте время подсчета доли отмененных рейсов по перевозчикам. Какой размер партиции оптимален и почему слишком мелкие и слишком крупные партиции плохи?

In [10]:
block_sizes = ["16MB", "64MB", "256MB"]
partition_counts = {}
dask_times = {}
dask_results = {}

for block_size in block_sizes:
    frame = dd.read_csv(
        FILES,
        blocksize=block_size,
        usecols=["OP_UNIQUE_CARRIER", "CANCELLED"],
        assume_missing=True,
    )
    partition_counts[block_size] = frame.npartitions

    def cancelled_share(current=frame):
        return current.groupby("OP_UNIQUE_CARRIER")["CANCELLED"].mean().compute()

    result, elapsed = timeit(cancelled_share, f"Dask, blocksize={block_size}")
    dask_results[block_size] = result
    dask_times[block_size] = elapsed

check_series_equal(dask_results)
print("Число партиций:", partition_counts)
optimal = min(dask_times, key=dask_times.get)
print(f"Оптимальный размер в этом запуске: {optimal} ({dask_times[optimal]:.2f} с).")
print("Слишком мелкие партиции создают много служебных задач и накладных расходов.")
print("Слишком крупные партиции хуже распределяют работу между ядрами и требуют больше памяти.")

Dask, blocksize=16MB: 1.19 с


Dask, blocksize=64MB: 1.27 с


Dask, blocksize=256MB: 1.05 с
Проверка пройдена: все результаты совпадают.
Число партиций: {'16MB': 43, '64MB': 11, '256MB': 11}
Оптимальный размер в этом запуске: 256MB (1.05 с).
Слишком мелкие партиции создают много служебных задач и накладных расходов.
Слишком крупные партиции хуже распределяют работу между ядрами и требуют больше памяти.


## Завершение сессии

In [11]:
spark.stop()